In [4]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

In [5]:
headers = headers = {
    "User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:150.0) Gecko/20100101 Firefox/150.0"
}

In [9]:
def get_top_scorers(headers, league, season):
    top_scorers = []

    url = f'https://www.transfermarkt.com/{league}/torschuetzenliste/wettbewerb/GB1/saison_id/{season}/altersklasse/alle/detailpos//page/1'
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content,'lxml')

    # --------------------------------------------------
    # Finding the last page
    # --------------------------------------------------
    pages_info = soup.find_all('div', {'class':'pager'})
    last_page_link = pages_info[0].find_all('li',{'class':'tm-pagination__list-item tm-pagination__list-item--icon-last-page'})
    last_page_number = last_page_link[0].find('a').get('href').split('/')[-1]


    # --------------------------------------------------
    # Getting information
    # --------------------------------------------------
    for n_page in range(1,int(last_page_number)+1):
        url = f'https://www.transfermarkt.com/{league}/torschuetzenliste/wettbewerb/GB1/saison_id/{season}/altersklasse/alle/detailpos//page/{n_page}'
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content,'lxml')
        
        # --------------------------------------------------
        # Extrating only the usefull information in the transfermarkt source page
        # --------------------------------------------------
        all_info = soup.find_all('table')
        # Transfermarkt separates information by index (odd, even)
        odd_info = all_info[1].find_all('tr',{'class':'odd'})
        even_info = all_info[1].find_all('tr',{'class':'even'})

        player_list = [odd_info,even_info]

        for player in player_list:
            for row in player:
                temp = []

                # --------------------------------------------------
                # Extracting data information
                # --------------------------------------------------
                # Creating a list containing only the main data points
                data = row.find_all('td',{'class':'zentriert'})

                # Extracting all relevant data and storing in different variables, mainly for better understanding
                pos = int(data[0].string)
                country = data[1].find('img').get('alt')
                age = int(data[2].string)
                name = data[4].find('a').get('title')
                matches = int(data[4].find('a').string)
                goals = int(data[5].find('a').string)

                # Some players scored for more than one club, that behaves differently in the transfermarkt source page
                try:team = data[3].find('a').get('title')
                except AttributeError: team = data[3].string
                
                # Creating the season key
                season_key = f'PL-{season}'

                # Gathering all the information for one player
                temp.append(season_key)
                temp.append(pos)
                temp.append(country)
                temp.append(age)
                temp.append(name)
                temp.append(team)
                temp.append(matches)
                temp.append(goals)

                # Appending the payer information in the main list
                top_scorers.append(temp)
    
    # Informing the headers of the list created
    head = (['season_id','pos','country','age','player_name','team','matches','goals'])
    top_scorers.insert(0,head)
    return top_scorers

In [11]:
league = 'premier-league'

headers = headers = {
    "User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:150.0) Gecko/20100101 Firefox/150.0"
}

test = get_top_scorers(headers,league,2025)
df_test = pd.DataFrame(test[1:],columns=test[0])
df_test.sort_values(by=['season_id','pos'],inplace=True, ignore_index=True)
display(df_test)

,season_id,pos,country,age,player_name,team,matches,goals
0,PL-2025,1,Norway,25,Erling Haaland,Manchester City,35,27
1,PL-2025,2,Brazil,24,Igor Thiago,Brentford FC,38,22
2,PL-2025,3,Ghana,26,Antoine Semenyo,for 2 clubs,37,17
3,PL-2025,4,England,30,Ollie Watkins,Aston Villa,37,16
4,PL-2025,5,Brazil,24,João Pedro,Chelsea FC,35,15
...,...,...,...,...,...,...,...,...
275,PL-2025,276,Denmark,25,Matt O'Riley,Brighton & Hove Albion,6,1
276,PL-2025,277,Portugal,23,Fábio Carvalho,Brentford FC,6,1
277,PL-2025,278,Italy,25,Lorenzo Lucca,Nottingham Forest,4,1
278,PL-2025,279,Wales,32,Ben Davies,Tottenham Hotspur,3,1
